### MC Methods - Noughts and Crosses

We simulate two players playing noughts and crosses by considering a sole agent who makes decisions under a policy that makes random moves. During an episode, a random legal move is made by the agent. In the proceeding episode, the agent makes another random move but switches sign. The first move of a game, is always noughts (the next is crosses and so on). Therefore, one can infer the player turn from the episode number deterministically. The result is a trajectory that is indistinguishable from two players making random moves. I represent the grid as a `3x3` matrix with zeros representing empty cells. Subsequently, `1` and `2` represent noughts ($O$) and crosses ($X$) respectively. The code is as follows,

In [2]:
# Dependencies,
import numpy as np

class DumbPlayer():

    def __init__(self):
        """Constructor method. The grid is created."""
        self.grid = np.zeros(shape=(3,3))
        self.last_state = True
        self.env_states = []

    def play_game(self):

        self.winner = None
        game_terminated = False
        while not game_terminated:

            # Selecting random free cell,
            empty_cell_idxs = np.argwhere(self.grid == 0)

            if len(empty_cell_idxs) == 0:
                game_terminated = True
                break
            
            selected_cell_idx = np.random.randint(low=0, high=len(empty_cell_idxs))
            selected_cell_idxs =  empty_cell_idxs[selected_cell_idx]

            # Drawing a nought or cross,
            self.grid[tuple(selected_cell_idxs)] = 1 if self.last_state else 2
            self.last_state = not self.last_state
                
            # Saving current environment state,
            self.env_states.append(self.grid.flatten())

            # Checking for a winner,
            self.winner = self._check_winner()
            if self.winner is not None:
                game_terminated = True

    def return_results(self):
        """Returns the episodes and reward of the simulation."""

        if self.winner == 0:
            reward = 1
        elif self.winner == 1:
            reward = -1
        else: 
            reward = 0
        
        return np.array(self.env_states), reward

    def _check_winner(self):
        """Returns 0 if noughts has won the game, 1 if noughts has lost and 2 if there is no winner."""

        # Checking rows,
        for row in self.grid:
            if np.all(row == 1):
                return 0
            if np.all(row == 2):
                return 1
    
        # Checking columns,
        for col in self.grid.T:
            if np.all(col == 1):
                return 0
            if np.all(col == 2):
                return 1

        # Checking diagonals,
        diag1 = np.diag(self.grid)
        diag2 = np.diag(np.fliplr(self.grid))

        if np.all(diag1 == 1) or np.all(diag2 == 1):
            return 0

        if np.all(diag1 == 2) or np.all(diag2 == 2):
            return 1

    def reset(self):
        """Resets the game environment."""
        self.grid = np.zeros(shape=(3,3))
        self.last_state = True
        self.env_states = []
        self.winner = None

Let us consider a single simulated game,

In [3]:
player = DumbPlayer()
player.play_game()
episodes, reward = player.return_results()
episodes, reward

(array([[1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 2., 0., 0., 0., 0.],
        [1., 1., 0., 0., 2., 0., 0., 0., 0.],
        [1., 1., 0., 2., 2., 0., 0., 0., 0.],
        [1., 1., 0., 2., 2., 0., 0., 1., 0.],
        [1., 1., 0., 2., 2., 0., 2., 1., 0.],
        [1., 1., 1., 2., 2., 0., 2., 1., 0.]]),
 1)

Now we play multiple games, 

In [8]:
# Number of games,
N = 10000

# Creating trajectories,
player = DumbPlayer()

trajectories = []
rewards = []
for i in range(N):
    player = DumbPlayer()
    player.play_game()
    episodes, reward = player.return_results()
    trajectories.append(episodes)
    rewards.append(reward)
    player.reset() # <-- Reset game.

Let us now implement MC evaluation. Our aim is to quantify the value of a given state $s$ without an interal model. Given a policy $\Pi$, MC methods computes,

$$
v_{\pi}(s) = E_{\Pi} [G_t \vert S_t = s] \approx \frac{1}{C(s)} \sum_{m=1}^M \sum_{\tau=0}^{T_m - 1} \mathcal{I}(s_\tau^m = s)g_{\tau}^m
$$

In our case, only rewards associated with terminal states are non-zero. This means that $r^{(m)}_{k} = 0$ for all $k \neq T_m$. As a consequence,

$$
g^{(m)}_{\tau} = r^{(m)}_{\tau + 1} + \gamma r^{(m)}_{\tau + 2} + + \gamma^2 r^{(m)}_{\tau + 3} + ... + \gamma^{T_m - 1 - \tau}  r^{(m)}_{T_m}
$$

$$
= \gamma^{T_m - 1 - \tau}  r^{(m)}_{T_m}
$$

To simplify things even further, we apply no discounting such that $\gamma = 1$ and,

$$
g^{(m)}_{\tau} = r^{(m)}_{T_m} = r_m
$$

With this, our computation of $v_{\pi}(s)$ is greatly reduced in complexity, 

$$
v_{\pi}(s) \approx \frac{1}{C(s)} \sum_{m=1}^M \sum_{\tau=0}^{T_m - 1} \mathcal{I}(s_\tau^m = s)g_{\tau}^m
$$

$$
= \frac{1}{C(s)} \sum_{m=1}^M \sum_{\tau=0}^{T_m - 1} \mathcal{I}(s_\tau^m = s)r_m
$$

$$
= \frac{1}{C(s)} \sum_{m=1}^M r_m
\underbrace{\left( \sum_{\tau=0}^{T_m - 1} \mathcal{I}(s_\tau^m = s) \right)}_{N_m(s)}
$$

where $N_m(s)$ is the number of times the state $s$ is occurs in the $m$-th trajectory. The counting function $C(s)$ is also the sum of $N_m(s)$ across all trajectories such that $C(s) = \sum_m N_m(s)$. As a result, our final calculation is: 

$$
\boxed{
v_{\pi}(s) \approx \frac{\sum_{m=1}^{M} r_m \, N_m(s)}{\sum_{m=1}^{M} N_m(s)}
}
$$

Implementing the equation in code is straight forward since $v_{\Pi}$(s) can be approximated as a weighted average of the rewards over the trajectories.

In [32]:
def compute_value(state):
    # Computing counting function and weighted reward,
    counting_func = 0
    weighted_rewards = 0
    for idx, trajectory in enumerate(trajectories):

        # Extracting terminal reward (of trajectory),
        reward = rewards[idx]
        
        # Counting occurance of state s in trajectory,
        N_state = 0
        for s in trajectory:
            if (s == state).all():
                N_state += 1

        weighted_rewards += reward*N_state
        counting_func += N_state # <-- Updating counting function.

    # Computing value,
    value = weighted_rewards/counting_func
    return value

For example, let us consider the state $s = [1, 2, 2, 1, 1, 2, 2, 1, 0]$ which corresponds to the state,

$$
s = 
\begin{bmatrix}
1 & 2 & 2 \\
1 & 1 & 2 \\
2 & 1 & 0
\end{bmatrix}

\Rightarrow \begin{bmatrix}
O & X & X \\
O & O & X \\
X & O & 
\end{bmatrix}

$$

Since we always start with noughts $O$, we can infer that it is their turn and that they are about to win the game (regardless of the action-selection rule because their is only a single legal move). Therefore, one can expect this particular state $s$ to have the value $v_{\pi}(s) = 1$ because the reward $r_m$ is always `+1` every time $s$ is present in a trajectory, as this state admits only one legal action, which deterministically leads to a terminal win for $O$.

In [33]:
s = [1, 2, 2, 1, 1, 2, 2, 1, 0]
print(f"v(s)={compute_value(state=s)}")

v(s)=1.0


Let us try another set of sanity checks. The state $s_1$ always results in a draw, $s_2$ always results in a loss, and $s_3$ is more ambiguous.

$$
s_1 =
\begin{bmatrix}
2 & 1 & 2 \\
2 & 1 & 1 \\
1 & 2 & 1
\end{bmatrix}
\Rightarrow
\begin{bmatrix}
X & O & X \\
X & O & O \\
O & X & O
\end{bmatrix}
\quad \text{(draw)}
$$

$$
s_2 =
\begin{bmatrix}
2 & 2 & 1 \\
1 & 2 & 1 \\
1 & 2 & 0
\end{bmatrix}
\Rightarrow
\begin{bmatrix}
X & X & O \\
O & X & O \\
O & X & \
\end{bmatrix}
\quad \text{(loss for O)}
$$

$$
s_3 =
\begin{bmatrix}
1 & 0 & 0 \\
0 & 2 & 0 \\
0 & 0 & 1
\end{bmatrix}
\Rightarrow
\begin{bmatrix}
O & \ & \  \\
\ & X & \  \\
\ & \ & O
\end{bmatrix}
\quad \text{(ambiguous)}
$$

In [37]:
# Defining the states,
s_1 = [2, 1, 2, 2, 1, 1, 1, 2, 1]
s_2 = [2, 2, 1, 1, 2, 1, 1, 2, 0]
s_3 = [1, 0, 0, 0, 2, 0, 0, 0, 1]

# Computing v(s_i),
print(f"v(s1)={compute_value(state=s_1)}")
print(f"v(s2)={compute_value(state=s_2)}")
print(f"v(s3)={compute_value(state=s_3)}")

v(s1)=0.0
v(s2)=-1.0
v(s3)=-0.14634146341463414


For $s_3$, the value $v_(s_3)$ infers that the position is slightly more advantageous for crosses $X$. 